# Fine-tuning de mBERT, LaBSE y XLM-R frente a sus baselines de embeddings congelados (complemento del OE4)

**Pregunta:** ¿reentrenar cada modelo con las oraciones reales del corpus mejora a su baseline del OE2 (embeddings congelados + Regresión Logística, protocolo B)? Y después de reentrenar, ¿sigue siendo mBERT el mejor?

Este cuaderno **generaliza el anterior** (que solo hacía mBERT): corre los tres modelos con el mismo protocolo. Si ya corriste mBERT con el cuaderno anterior, sus resultados guardados en tu Drive **se reutilizan** (no se vuelve a entrenar).

**Es un experimento exploratorio y separado.** No cambia ningún resultado del OE2 ni del OE3.

| Elemento | Igual que en el OE2 | Diferente (a propósito) |
|---|---|---|
| Corpus | 700 oraciones, 7 intenciones, texto normalizado | |
| Particiones | Las 5 particiones congeladas (`folds_fijos.csv`) | |
| Oraciones de prueba | Nunca se usan para elegir nada | |
| Modelos | `bert-base-multilingual-cased`, `sentence-transformers/LaBSE`, `xlm-roberta-base` (los mismos del OE2) | **Se reentrenan** (antes estaban congelados) |
| Clasificador | | Capa de clasificación entrenada junto con el modelo |
| Hiperparámetros | Se eligen sin tocar la prueba (análogo al protocolo B) | Tasa de aprendizaje y épocas, con validación interna del entrenamiento |
| Repeticiones | | 3 semillas por modelo |
| Comparación | F1 macro agrupado de las 700 predicciones, IC 95% bootstrap y **diferencia pareada** | |

**Misma rejilla de hiperparámetros para los tres modelos**, para que la comparación sea justa. Si algún modelo elige el borde de la rejilla (por ejemplo 10 épocas), el resultado de ese modelo puede ser conservador; el cuaderno lo avisa.

**No usa la API de Claude (no gasta dinero) y no agrega texto sintético.**

**Tiempo estimado (T4 de Colab, no medido):** mBERT ~60 min (ya hecho si usaste el cuaderno anterior); LaBSE y XLM-R son modelos más grandes (vocabulario enorme), así que calcula ~1.5 a 2 horas cada uno. Todo se guarda en tu Drive a medida que avanza: si Colab se desconecta, vuelve a correr desde la celda de configuración y retoma.

**Uso:** Entorno de ejecución → GPU (T4). Corre las celdas en orden. Para comprobar que todo funciona, pon `PRUEBA = True` en la configuración (unos minutos por modelo).

## 0. Montar Google Drive (aquí queda guardado el avance y los resultados)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. Clonar tu repositorio

In [ ]:
import os
GITHUB_USUARIO = "LhiaHC"

%cd /content
!rm -rf /content/shiwilu-tesis
!git clone https://github.com/{GITHUB_USUARIO}/shiwilu-tesis.git
%cd /content/shiwilu-tesis
!git log --oneline -3

REPO = '/content/shiwilu-tesis'
requeridos = [
    f'{REPO}/corpus/corpus_shiwilu_final.csv',
    f'{REPO}/oe2_aumento_de_datos/particiones/folds_fijos.csv',
    f'{REPO}/oe2_aumento_de_datos/evaluacion/resultados/cv_interna/validacion_cruzada_predicciones_sin_puntuacion_cv_interna_c120.csv',
    f'{REPO}/shiwilu/clasificacion.py',
]
faltan = [r for r in requeridos if not os.path.exists(r)]
assert not faltan, 'Faltan archivos en el clon (haz git push de lo nuevo y vuelve a correr esta celda): ' + ', '.join(faltan)
print('OK')

## 2. Instalar dependencias y comprobar la GPU

`torchao` se desinstala a propósito: Colab trae una versión vieja que choca con `peft` (error «Found an incompatible version of torchao») y este cuaderno no la usa.
Si ese error reaparece, corre solo esta celda otra vez y luego **Entorno de ejecución → Reiniciar sesión** (no pierdes nada: el avance está en Drive).

In [ ]:
!pip install -q transformers peft accelerate scikit-learn sentencepiece
!pip uninstall -y -q torchao
import torch
print('GPU disponible:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'SIN GPU (cambia el tipo de entorno de ejecucion)')

## 3. Configuración (corre esta celda en cada sesión)

- `MODELOS`: los tres modelos del OE2 con sus identificadores de Hugging Face.
- `MODELOS_A_CORRER`: cuáles entrenar en esta sesión. Por defecto `["xlmr_rejilla2"]`: XLM-R con una rejilla corregida (tasas de aprendizaje más bajas y más épocas), porque con la rejilla común no convergió. mBERT, LaBSE y XLM-R con la rejilla común ya están hechos y se reutilizan.
- `VARIANTES`: `["completo"]` (todos los pesos). LoRA quedó fuera: con la rejilla probada no fue concluyente y no es necesaria para esta comparación.
- `SEMILLAS`, `LRS`, `EPOCAS`, `N_INTERNOS`: iguales para mBERT y LaBSE. `GRILLAS_POR_MODELO` da a XLM-R una rejilla propia (declarada a posteriori y elegida, como siempre, solo con validación interna).
- `PRUEBA = True`: corrida mínima para comprobar que todo funciona (1 partición, 1 semilla, 1 combinación, 2 épocas).

In [ ]:
import os, re, sys, math, time, random, glob, shutil
import numpy as np, pandas as pd, torch

MODELOS = {"mbert": "bert-base-multilingual-cased",
           "labse": "sentence-transformers/LaBSE",
           "xlmr": "xlm-roberta-base",
           "xlmr_rejilla2": "xlm-roberta-base"}   # mismo XLM-R con una rejilla corregida (ver GRILLAS_POR_MODELO)
MODELOS_A_CORRER = ["xlmr_rejilla2"]          # mBERT, LaBSE y XLM-R (rejilla 1) ya estan hechos; sus resultados se reutilizan
VARIANTES = ["completo"]
SEMILLAS = [0, 1, 2]
FOLDS = [0, 1, 2, 3, 4]
LRS = {"completo": [2e-5, 5e-5, 1e-4]}
EPOCAS = [5, 10]
# Rejilla propia de un modelo. XLM-R no convergio con la rejilla comun (a 1e-4 colapso: F1 de validacion ~0.09) y necesita tasas mas bajas y mas epocas.
# Es un ajuste declarado a posteriori y se elige, como siempre, solo con validacion interna.
GRILLAS_POR_MODELO = {"xlmr_rejilla2": {"lrs": [1e-5, 2e-5, 3e-5], "epocas": [10, 15]}}
N_INTERNOS = 2
BATCH = 16
MAX_LEN = 64
REPETICIONES_BOOTSTRAP = 2000
PRUEBA = False                                 # True = comprobacion rapida

if PRUEBA:
    MODELOS_A_CORRER, VARIANTES, SEMILLAS, FOLDS, LRS, EPOCAS, N_INTERNOS = ["mbert"], ["completo"], [0], [0], {"completo": [5e-5]}, [2], 1

SUFIJO = "_prueba" if PRUEBA else ""
DIR_DRIVE = "/content/drive/MyDrive/shiwilu_finetuning" + SUFIJO       # avance y resultados (sobrevive a desconexiones)
DIR_RUNS = DIR_DRIVE + "/corridas"
os.makedirs(DIR_RUNS, exist_ok=True)
DEV = "cuda" if torch.cuda.is_available() else "cpu"
USAR_FP16 = DEV == "cuda"

# Compatibilidad: el cuaderno anterior guardaba mBERT sin el nombre del modelo en el archivo (p. ej. completo_s0_f0.csv).
migrados = 0
for ruta in glob.glob(f"{DIR_RUNS}/*.csv"):
    nombre = os.path.basename(ruta)
    if re.match(r"^(seleccion_)?(completo|lora)_", nombre) or re.match(r"^(completo|lora)_s\d_f\d\.csv$", nombre):
        nuevo = re.sub(r"^(seleccion_)?(completo|lora)_", lambda m: f"{m.group(1) or ''}mbert_{m.group(2)}_", nombre)
        if not os.path.exists(f"{DIR_RUNS}/{nuevo}"):
            os.rename(ruta, f"{DIR_RUNS}/{nuevo}"); migrados += 1
print("Dispositivo:", DEV, "| modelos a correr:", MODELOS_A_CORRER, "| variantes:", VARIANTES, "| semillas:", SEMILLAS, "| particiones:", FOLDS)
if migrados:
    print(f"Se renombraron {migrados} archivos de mBERT del cuaderno anterior (ahora llevan el prefijo 'mbert_').")

## 4. Cargar los datos (los mismos del OE2)

Mismo corpus, mismo texto normalizado y mismas particiones congeladas que en los 12 experimentos. `pos` es la posición de cada oración en el corpus (0 a 699), igual que en los archivos de predicciones del OE2, lo que permite la comparación pareada.

In [ ]:
sys.path.insert(0, REPO)
from shiwilu.clasificacion import cargar_corpus, cargar_corpus_normalizado, cargar_folds, _normalizar_estricto
from shiwilu.taxonomia import INTENCIONES

CLASES = sorted(INTENCIONES)                                  # AFI, DES, EMO, NEG, PRG, REQUEST, SAL
corpus = cargar_corpus()
assert corpus.index.equals(pd.RangeIndex(len(corpus)))
texto = cargar_corpus_normalizado()["shiwilu"].astype(str).to_numpy()
y = corpus["intencion"].to_numpy()
Y_IDX = np.array([CLASES.index(c) for c in y])
folds = cargar_folds(corpus)
grupos = np.array([_normalizar_estricto(t) for t in corpus["shiwilu"]])
print(len(corpus), "oraciones |", len(CLASES), "intenciones | tamanos de prueba por particion:", [int((folds == f).sum()) for f in range(5)])

def f1_macro_idx(yv, pv, K=len(CLASES)):
    cm = np.bincount(yv * K + pv, minlength=K * K).reshape(K, K)
    tp = np.diag(cm); fp = cm.sum(0) - tp; fn = cm.sum(1) - tp
    den = 2 * tp + fp + fn
    return float(np.where(den > 0, 2 * tp / np.maximum(den, 1), 0.0).mean())

def f1_por_clase(yv, pv, K=len(CLASES)):
    cm = np.bincount(yv * K + pv, minlength=K * K).reshape(K, K)
    tp = np.diag(cm); fp = cm.sum(0) - tp; fn = cm.sum(1) - tp
    den = 2 * tp + fp + fn
    return np.where(den > 0, 2 * tp / np.maximum(den, 1), 0.0)

## 5. Funciones de entrenamiento

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.model_selection import StratifiedGroupKFold

TOKS = {}
def tokens_de(modelo):
    # cada modelo tiene su propio tokenizador: se tokeniza una vez y se reutiliza
    if modelo not in TOKS:
        tok = AutoTokenizer.from_pretrained(MODELOS[modelo])
        enc = tok(list(texto), padding=True, truncation=True, max_length=MAX_LEN, return_tensors="pt")
        TOKS[modelo] = (enc["input_ids"], enc["attention_mask"])
        print(f"[{modelo}] ancho de secuencia (piezas): {enc['input_ids'].shape[1]}")
    return TOKS[modelo]

def fijar_semilla(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def entrenar_y_predecir(modelo, idx_tr, idx_evals, lr, epocas, semilla, variante):
    # entrena con las oraciones idx_tr y devuelve las predicciones (indices de clase) de cada conjunto de idx_evals
    IDS, MASK = tokens_de(modelo)
    fijar_semilla(semilla)
    net = AutoModelForSequenceClassification.from_pretrained(
        MODELOS[modelo], num_labels=len(CLASES), id2label=dict(enumerate(CLASES)), label2id={c: i for i, c in enumerate(CLASES)})
    if variante == "lora":
        from peft import LoraConfig, get_peft_model
        net = get_peft_model(net, LoraConfig(task_type="SEQ_CLS", r=16, lora_alpha=32, lora_dropout=0.1, target_modules=["query", "value"]))
    net.to(DEV)
    params = [p for p in net.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=0.01)
    pasos = epocas * math.ceil(len(idx_tr) / BATCH)
    sch = get_linear_schedule_with_warmup(opt, int(0.1 * pasos), pasos)
    scaler = torch.amp.GradScaler("cuda", enabled=USAR_FP16)
    g = torch.Generator().manual_seed(semilla)
    ytr = torch.as_tensor(Y_IDX[idx_tr])
    net.train()
    for _ in range(epocas):
        perm = torch.randperm(len(idx_tr), generator=g)
        for k in range(0, len(perm), BATCH):
            b = perm[k:k + BATCH]
            ii = torch.as_tensor(idx_tr[b.numpy()])
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USAR_FP16):
                salida = net(input_ids=IDS[ii].to(DEV), attention_mask=MASK[ii].to(DEV), labels=ytr[b].to(DEV))
            opt.zero_grad(set_to_none=True)
            scaler.scale(salida.loss).backward()
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(params, 1.0)
            scaler.step(opt); scaler.update(); sch.step()
    net.eval(); salidas = []
    with torch.no_grad():
        for idx in idx_evals:
            pr = []
            for k in range(0, len(idx), 64):
                ii = torch.as_tensor(idx[k:k + 64])
                with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USAR_FP16):
                    lg = net(input_ids=IDS[ii].to(DEV), attention_mask=MASK[ii].to(DEV)).logits
                pr.append(lg.argmax(-1).cpu().numpy())
            salidas.append(np.concatenate(pr))
    del net, opt
    if DEV == "cuda":
        torch.cuda.empty_cache()
    return salidas

def correr_particion(modelo, f, semilla, variante):
    # elige (lr, epocas) con validacion interna del entrenamiento y predice la prueba de la particion f
    idx_test = np.where(folds == f)[0]
    idx_pool = np.where(folds != f)[0]
    cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=semilla)
    internos = list(cv.split(idx_pool, Y_IDX[idx_pool], groups=grupos[idx_pool]))[:N_INTERNOS]
    candidatos = []
    grilla = GRILLAS_POR_MODELO.get(modelo, {})
    for lr in grilla.get("lrs", LRS[variante]):
        for ep in grilla.get("epocas", EPOCAS):
            f1s = []
            for tr, va in internos:
                p = entrenar_y_predecir(modelo, idx_pool[tr], [idx_pool[va]], lr, ep, semilla, variante)[0]
                f1s.append(f1_macro_idx(Y_IDX[idx_pool[va]], p))
            candidatos.append({"lr": lr, "epocas": ep, "f1_validacion_interna": float(np.mean(f1s))})
    cand = pd.DataFrame(candidatos)
    mejor = max(candidatos, key=lambda c: (c["f1_validacion_interna"], -c["epocas"], -c["lr"]))   # desempate: menos epocas, menor lr
    pred = entrenar_y_predecir(modelo, idx_pool, [idx_test], mejor["lr"], mejor["epocas"], semilla, variante)[0]
    salida = pd.DataFrame({"modelo": modelo, "variante": variante, "semilla": semilla, "fold": f, "pos": idx_test, "real": y[idx_test],
                           "prediccion": [CLASES[i] for i in pred], "lr": mejor["lr"], "epocas": mejor["epocas"],
                           "f1_validacion_interna": mejor["f1_validacion_interna"]})
    return salida, cand

## 6. Correr los experimentos (retoma sola si Colab se desconecta)

Para cada modelo, variante, semilla y partición se guarda un archivo en tu Drive en cuanto termina. Si vuelves a correr la celda, salta lo que ya existe.
Cada partición imprime la combinación elegida y su F1 de validación interna (no es el F1 de prueba).

In [ ]:
t_ini = time.time()
for modelo in MODELOS_A_CORRER:
    for variante in VARIANTES:
        for semilla in SEMILLAS:
            for f in FOLDS:
                ruta = f"{DIR_RUNS}/{modelo}_{variante}_s{semilla}_f{f}.csv"
                if os.path.exists(ruta):
                    print(f"[omitido] {modelo} {variante} semilla {semilla} particion {f}")
                    continue
                t0 = time.time()
                salida, cand = correr_particion(modelo, f, semilla, variante)
                cand.to_csv(f"{DIR_RUNS}/seleccion_{modelo}_{variante}_s{semilla}_f{f}.csv", index=False)
                salida.to_csv(ruta, index=False)
                f1_prueba = f1_macro_idx(Y_IDX[salida.pos.to_numpy()], np.array([CLASES.index(c) for c in salida.prediccion]))
                print(f"{modelo} {variante} semilla {semilla} particion {f}: elegido lr={salida.lr.iloc[0]:g}, epocas={salida.epocas.iloc[0]}, "
                      f"F1 val. interna={salida.f1_validacion_interna.iloc[0]:.3f} | F1 prueba de esta particion={f1_prueba:.3f} | {time.time() - t0:.0f} s")
print(f"\nTiempo total de esta sesion: {(time.time() - t_ini) / 60:.1f} min")

## 7. Resultados y comparaciones pareadas

Para cada modelo reentrenado se une la prueba de las 5 particiones (700 predicciones) y se calcula el **F1 macro agrupado**, igual que en el OE2. Se hacen tres comparaciones, todas **pareadas** (se remuestrean las mismas oraciones y se promedia el F1 de las semillas):

1. **Cada modelo reentrenado contra su propio baseline congelado** (el del OE2, protocolo B).
2. **Los modelos reentrenados entre sí** (¿sigue ganando mBERT?).
3. **Aviso de rejilla:** cuántas veces cada modelo eligió el borde de la rejilla (si es casi siempre, el resultado de ese modelo puede ser conservador).

In [ ]:
base_todo = pd.read_csv(f"{REPO}/oe2_aumento_de_datos/evaluacion/resultados/cv_interna/validacion_cruzada_predicciones_sin_puntuacion_cv_interna_c120.csv")
P_BASE = {}
for m in ["mbert", "labse", "xlmr"]:
    b = base_todo[(base_todo.modelo == m) & (base_todo.tecnica == "sin_aumento")].sort_values("pos")
    assert len(b) == 700 and (b.real.to_numpy() == y).all(), f"el baseline de {m} no esta alineado con el corpus"
    P_BASE[m] = np.array([CLASES.index(c) for c in b.prediccion])
print("Baselines congelados (OE2, protocolo B):", {m: round(f1_macro_idx(Y_IDX, p), 3) for m, p in P_BASE.items()}, "\n")

def cargar(modelo, variante):
    # devuelve {semilla: (posiciones, predicciones)}; exige las 700 oraciones salvo en modo PRUEBA
    preds = {}
    for semilla in SEMILLAS:
        partes = [pd.read_csv(r) for f in FOLDS for r in [f"{DIR_RUNS}/{modelo}_{variante}_s{semilla}_f{f}.csv"] if os.path.exists(r)]
        if not partes:
            continue
        d = pd.concat(partes).sort_values("pos")
        if len(d) == 700 or PRUEBA:
            preds[semilla] = (d.pos.to_numpy(), np.array([CLASES.index(c) for c in d.prediccion]))
    return preds

rng = np.random.default_rng(42)
disponibles = {}
for modelo in MODELOS:
    for variante in VARIANTES:
        pr = cargar(modelo, variante)
        if pr:
            disponibles[(modelo, variante)] = pr
print("Resultados disponibles:", [f"{m}/{v} ({len(p)} semillas)" for (m, v), p in disponibles.items()])

filas, por_clase = [], {}
for (modelo, variante), preds in disponibles.items():
    pos = next(iter(preds.values()))[0]
    yv, pb = Y_IDX[pos], P_BASE[modelo.split("_")[0]][pos]
    P = np.stack([p for (_, p) in preds.values()])
    f1s = [f1_macro_idx(yv, p) for p in P]
    n = len(yv); difs, medias = [], []
    for _ in range(REPETICIONES_BOOTSTRAP):
        i = rng.integers(0, n, n)
        f1_ft = np.mean([f1_macro_idx(yv[i], p[i]) for p in P])
        difs.append(f1_ft - f1_macro_idx(yv[i], pb[i])); medias.append(f1_ft)
    lo, hi = np.percentile(difs, [2.5, 97.5])
    filas.append({"modelo": modelo, "variante": variante, "oraciones": n, "semillas": len(P), "f1_baseline_congelado": f1_macro_idx(yv, pb),
                  "f1_finetuning_media": float(np.mean(f1s)), "f1_sd_entre_semillas": float(np.std(f1s)), "f1_minimo": min(f1s), "f1_maximo": max(f1s),
                  "ic95_bajo": np.percentile(medias, 2.5), "ic95_alto": np.percentile(medias, 97.5),
                  "dif_vs_baseline": float(np.mean(f1s) - f1_macro_idx(yv, pb)), "dif_ic95_bajo": lo, "dif_ic95_alto": hi, "distinguible_de_cero": bool(lo > 0 or hi < 0)})
    por_clase[f"{modelo} congelado"] = f1_por_clase(yv, pb)
    por_clase[f"{modelo} fine-tuning {variante}"] = np.mean([f1_por_clase(yv, p) for p in P], axis=0)
RES = pd.DataFrame(filas)
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30)
print("\n=== 1. Cada modelo reentrenado frente a su baseline congelado ===")
print(RES.round(3).to_string(index=False))
print("\nF1 por intencion (promedio de semillas):")
print(pd.DataFrame(por_clase, index=CLASES).round(3).to_string())

# 2. modelos reentrenados entre si
cruce = []
claves = list(disponibles.keys())
for a in range(len(claves)):
    for b in range(a + 1, len(claves)):
        (ma, va), (mb, vb) = claves[a], claves[b]
        pos = np.intersect1d(next(iter(disponibles[claves[a]].values()))[0], next(iter(disponibles[claves[b]].values()))[0])
        yv = Y_IDX[pos]
        Pa = np.stack([p[np.isin(ps, pos)] for (ps, p) in disponibles[claves[a]].values()])
        Pb = np.stack([p[np.isin(ps, pos)] for (ps, p) in disponibles[claves[b]].values()])
        difs = []
        for _ in range(REPETICIONES_BOOTSTRAP):
            i = rng.integers(0, len(yv), len(yv))
            difs.append(np.mean([f1_macro_idx(yv[i], p[i]) for p in Pa]) - np.mean([f1_macro_idx(yv[i], p[i]) for p in Pb]))
        lo, hi = np.percentile(difs, [2.5, 97.5])
        cruce.append({"a": f"{ma} fine-tuning", "b": f"{mb} fine-tuning", "dif_a_menos_b": float(np.mean([f1_macro_idx(yv, p) for p in Pa]) - np.mean([f1_macro_idx(yv, p) for p in Pb])),
                      "ic95_bajo": lo, "ic95_alto": hi, "distinguible_de_cero": bool(lo > 0 or hi < 0)})
CRUCE = pd.DataFrame(cruce)
print("\n=== 2. Modelos reentrenados entre si (diferencia pareada) ===")
print(CRUCE.round(3).to_string(index=False) if len(CRUCE) else "Hace falta tener al menos dos modelos reentrenados.")

# 3. aviso de rejilla
print("\n=== 3. Aviso de rejilla: veces que se eligio el borde (maximo de epocas o de tasa de aprendizaje) ===")
for (modelo, variante) in disponibles:
    elegidos = pd.concat([pd.read_csv(r) for r in sorted(glob.glob(f"{DIR_RUNS}/{modelo}_{variante}_s*_f*.csv"))])[["semilla", "fold", "lr", "epocas"]].drop_duplicates(["semilla", "fold"])
    g = GRILLAS_POR_MODELO.get(modelo, {})
    max_ep, max_lr = max(g.get("epocas", EPOCAS)), max(g.get("lrs", LRS[variante]))
    borde_ep = int((elegidos.epocas == max_ep).sum()); borde_lr = int((elegidos.lr == max_lr).sum())
    print(f"{modelo} {variante}: epocas={max_ep} en {borde_ep}/{len(elegidos)} corridas | lr={max_lr:g} en {borde_lr}/{len(elegidos)} corridas")

## 8. Guardar y descargar los resultados

Los CSV se guardan en tu Drive y en `finetuning_resultados.zip`, que se descarga. Para llevarlos al repositorio, copia el contenido a `oe4_sintesis/finetuning/resultados/` (sin hacer commit hasta revisarlo).

In [ ]:
DIR_OUT = f"{DIR_DRIVE}/resultados"
os.makedirs(DIR_OUT, exist_ok=True)
if len(RES):
    RES.to_csv(f"{DIR_OUT}/finetuning_vs_baseline_todos.csv", index=False)
    pd.DataFrame(por_clase, index=CLASES).to_csv(f"{DIR_OUT}/f1_por_intencion_todos.csv")
if len(CRUCE):
    CRUCE.to_csv(f"{DIR_OUT}/finetuning_comparacion_entre_modelos.csv", index=False)
for (modelo, variante) in disponibles:
    partes = [pd.read_csv(r) for r in sorted(glob.glob(f"{DIR_RUNS}/{modelo}_{variante}_s*_f*.csv"))]
    pd.concat(partes).to_csv(f"{DIR_OUT}/predicciones_{modelo}_{variante}.csv", index=False)
    sel = [pd.read_csv(r).assign(archivo=os.path.basename(r)) for r in sorted(glob.glob(f"{DIR_RUNS}/seleccion_{modelo}_{variante}_*.csv"))]
    if sel:
        pd.concat(sel).to_csv(f"{DIR_OUT}/seleccion_hiperparametros_{modelo}_{variante}.csv", index=False)
shutil.make_archive("/content/finetuning_resultados", "zip", DIR_OUT)
print("Guardado en:", DIR_OUT, "|", sorted(os.listdir(DIR_OUT)))
try:
    from google.colab import files
    files.download("/content/finetuning_resultados.zip")
except Exception as e:
    print("Descarga manual desde Drive:", e)

## Cómo leer el resultado (y qué decir en el informe)

- **Tabla 1:** `dif_vs_baseline` es el F1 del modelo reentrenado (promedio de semillas) menos el de su baseline congelado. Es *distinguible de cero* solo si su intervalo **no incluye el 0**.
- **Tabla 2:** compara los modelos reentrenados entre sí. Responde si mBERT sigue siendo el mejor después de reentrenar, o si LaBSE o XLM-R lo alcanzan.
- **Aviso de rejilla:** si un modelo eligió 10 épocas (o la tasa más alta) casi siempre, probablemente podía mejorar con una rejilla más amplia; su resultado es conservador.
- **`f1_sd_entre_semillas`:** si es grande (más de 0.02), el fine-tuning de ese modelo es inestable y el promedio debe leerse con cautela.
- Las diferencias menores a unos 0.01 no se interpretan, como en el resto de la tesis.

**Referencia útil:** el clasificador de n-gramas de letras alcanza 0.751 con estas mismas particiones; sirve para ubicar los resultados (no está calculado en este cuaderno).

**Lo que este experimento no concluye:** que los modelos «entiendan» el shiwilu, ni que valga para otras lenguas. Solo compara usar cada modelo congelado o reentrenado sobre este corpus.